In [ ]:
import pandas as p
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score


In [ ]:
restaurantes = p.read_json("rest.txt")

In [ ]:
restaurantes["Cozinha1"] = restaurantes["Cozinha"].fillna('').str.split(',', n=1).str[0]

In [ ]:
restaurantes["Cozinha2"] = restaurantes["Cozinha"].fillna('').str.split(',', n=1).str[1]

In [ ]:
restaurantes["Bairro"] = restaurantes["Endereco"].fillna('').str.split(', ', n=1).str[1]

In [ ]:
restaurantes["Bairro"] = restaurantes["Bairro"].fillna('').str.split(' -', n=1).str[0]

In [ ]:
restaurantes["Bairro"] = restaurantes["Bairro"].fillna('').str.split(' ', n=1).str[1]

In [ ]:
restaurantes["Almoco"] = restaurantes["Valor"].fillna('').str.split('- Almo?o: ',n=1).str[1]

In [ ]:
restaurantes["Almoco"] = restaurantes["Almoco"].fillna('').str.split(' - Jantar:',n=1).str[0]

In [ ]:
restaurantes["Jantar"] = restaurantes["Valor"].fillna('').str.split(' - Jantar: ',n=1).str[1]

In [ ]:
restaurantes["Menu"] = restaurantes["Valor"].fillna('').str.split('Menu ',n=1).str[1]

In [ ]:
restaurantes["Menu"] = restaurantes["Menu"].fillna('').str.split(' -',n=1).str[0]

In [ ]:
restaurantes = restaurantes[['Nome', 'Cozinha1', 'Cozinha2', 'Bairro', 'Almoco', 'Jantar', 'Menu']]

In [ ]:
restaurantes

In [ ]:
temp1 = p.DataFrame(restaurantes['Cozinha1'].drop_duplicates())
temp1['Cozinha'] = temp1['Cozinha1']
temp1 = temp1.drop('Cozinha1', axis=1)
temp2 = p.DataFrame(restaurantes['Cozinha2'].drop_duplicates())
temp2['Cozinha'] = temp2['Cozinha2']
temp2 = temp2.drop('Cozinha2', axis=1)
temp = p.concat([temp1, temp2], ignore_index=True)
temp = p.DataFrame(temp['Cozinha'].drop_duplicates())
temp


In [ ]:
temp = p.DataFrame(restaurantes['Bairro'].drop_duplicates())
temp = temp.reset_index()
temp

In [ ]:
restaurantes.describe(include='all')

In [ ]:
frequencia = p.DataFrame()
frequencia['Menu'] = restaurantes['Menu'].value_counts(normalize=True).mul(100).round(2)
frequencia


In [ ]:
def cramers_v(confusion_matrix):
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    i, j = confusion_matrix.shape
    return (chi2 / (n * (min(i, j) - 1)))**0.5

In [ ]:
tabela_contingencia = p.crosstab(restaurantes['Bairro'], restaurantes['Cozinha1'])

v_cramer = cramers_v(tabela_contingencia)
print(f"  V de Cramer (calculado): {v_cramer:.2f}")

In [ ]:
tabela_contingencia = p.crosstab(restaurantes['Bairro'], restaurantes['Menu'])

v_cramer = cramers_v(tabela_contingencia)
print(f"  V de Cramer (calculado): {v_cramer:.2f}")

In [ ]:
tabela_contingencia = p.crosstab(restaurantes['Cozinha1'], restaurantes['Menu'])

v_cramer = cramers_v(tabela_contingencia)
print(f"  V de Cramer (calculado): {v_cramer:.2f}")

In [ ]:
print("\n--- Sumariza??o com Tabela de Conting?ncia (Crosstab) ---")
# Exemplo 1: Cor_Favorita vs Tipo_Filme
crosstab_filme_cor = p.crosstab(restaurantes['Menu'], restaurantes['Bairro'])
print("\nTabela de Conting?ncia: Cor_Favorita vs Tipo_Filme\n", crosstab_filme_cor)

In [ ]:
# Para obter propor??es na crosstab (ex: propor??o de cada linha)
crosstab_filme_cor_prop_linha = p.crosstab(restaurantes['Menu'], restaurantes['Bairro'], normalize='index').mul(100).round(2)
print("\nTabela de Conting?ncia (Propor??o por Linha):\n", crosstab_filme_cor_prop_linha)

In [ ]:
# Heatmap da Tabela de Conting?ncia (para vizualizar associa??es)
plt.figure(figsize=(12, 6))
sns.heatmap(crosstab_filme_cor, annot=True, fmt='d', cmap='Blues', linewidths=.5)
plt.title('Heatmap: Cor Favorita vs Tipo de Filme')
plt.xlabel('Tipo de Filme')
plt.ylabel('Cor Favorita')
plt.show()

In [ ]:
restaurantes.to_csv("restaurantes.csv", index=False)